# 02 · Bronze → Silver: clean and standardize the raw files

| Source (Files/raw) | Issues handled | Silver table |
|---|---|---|
| `erp/sales_invoices.csv` | duplicated export rows, lower-case / padded customer codes, text types | `silver_sales_invoices` |
| `erp/gl_cost_postings.csv` | Polish ERP format: `;` separator, decimal comma, `dd.mm.yyyy`, Polish headers | `silver_gl_costs` |
| `budget/budget_2025_2026.xlsx` | months as columns → unpivot to one row per month | `silver_budget_sales`, `silver_budget_costs`, `silver_budget_assumptions` |
| `master_data/*.csv` | trimming, types | `silver_products`, `silver_customers`, `silver_cost_centers`, `silver_accounts` |
| `bronze_nbp_fx_rates` | no rates on weekends/holidays; invoices use the rate from the **previous business day** (Polish VAT/CIT rule) | `silver_fx_rates_daily` |

**Requires:** default lakehouse `lh_fpa` attached, raw files uploaded to `Files/raw`, notebook 01 run.

In [ ]:
import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F

SPARK_FILES = "Files/raw"                       # path used by Spark
LOCAL_FILES = "/lakehouse/default/Files/raw"    # same folder seen by pandas (File API)
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def save_table(df, name: str) -> None:
    """Overwrite a Delta table in the attached lakehouse and print its row count."""
    (df.write.mode("overwrite").format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(name))
    print(f"✔ {name}: {spark.read.table(name).count():,} rows")

## 1. Sales invoices

In [ ]:
sales_raw = (spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(f"{SPARK_FILES}/erp/sales_invoices.csv"))

sales = (sales_raw
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("invoice_line", F.col("invoice_line").cast("int"))
    .withColumn("invoice_date", F.to_date("invoice_date", "yyyy-MM-dd"))
    .withColumn("quantity_t", F.col("quantity_t").cast("decimal(18,3)"))
    .withColumn("unit_price", F.col("unit_price").cast("decimal(18,4)"))
    .withColumn("net_amount", F.col("net_amount").cast("decimal(18,2)"))
    .dropDuplicates(["invoice_no", "invoice_line"]))

# data quality checks
n_raw, n_clean = sales_raw.count(), sales.count()
bad_amount = sales.filter(F.abs(F.col("quantity_t") * F.col("unit_price") - F.col("net_amount")) > 1).count()
print(f"Raw rows: {n_raw:,} | after de-duplication: {n_clean:,} | removed duplicates: {n_raw - n_clean}")
print(f"Lines where quantity × price ≠ amount: {bad_amount}")
assert sales.filter(F.col("invoice_date").isNull()).count() == 0, "Unparsed invoice dates"

save_table(sales, "silver_sales_invoices")

## 2. General ledger cost postings (Polish ERP export)

In [ ]:
gl_raw = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(f"{SPARK_FILES}/erp/gl_cost_postings.csv"))

gl = gl_raw.select(
    F.col("Nr_dokumentu").alias("document_no"),
    F.to_date("Data_ksiegowania", "dd.MM.yyyy").alias("posting_date"),
    F.trim("Konto").alias("account"),
    F.upper(F.trim("MPK")).alias("cost_center"),
    # "1 234 567,89" -> 1234567.89 (removes normal and non-breaking spaces, decimal comma -> dot)
    F.regexp_replace(F.regexp_replace("Kwota", r"[\s ]", ""), ",", ".").cast("decimal(18,2)").alias("amount_pln"),
    F.col("Opis").alias("description"),
)

assert gl.filter(F.col("posting_date").isNull() | F.col("amount_pln").isNull()).count() == 0, "Parsing problem in GL file"
save_table(gl, "silver_gl_costs")

## 3. Budget workbook (Excel, months as columns → unpivot)

In [ ]:
xlsx = f"{LOCAL_FILES}/budget/budget_2025_2026.xlsx"
month_no = {m: i for i, m in enumerate(MONTHS, start=1)}


def unpivot_months(pdf: pd.DataFrame, id_cols: list[str], value_name: str) -> pd.DataFrame:
    long = pdf.melt(id_vars=id_cols, value_vars=MONTHS, var_name="month_name", value_name=value_name)
    long["month_start"] = pd.to_datetime(pd.DataFrame(
        {"year": long["Year"], "month": long["month_name"].map(month_no), "day": 1})).dt.date
    return long.drop(columns=["month_name"])


# Sales budget: volume (t) per month + annual price per t
vol = unpivot_months(pd.read_excel(xlsx, sheet_name="Sales_Volume"), ["Year", "Product_ID", "Market"], "quantity_t")
price = pd.read_excel(xlsx, sheet_name="Sales_Price")
budget_sales = vol.merge(price, on=["Year", "Product_ID", "Market"], how="left", validate="many_to_one")
budget_sales = budget_sales.rename(columns={
    "Year": "year", "Product_ID": "product_id", "Market": "market",
    "Currency": "currency", "Price_per_t": "price_per_t"})
assert budget_sales["price_per_t"].notna().all(), "Missing budget price"

# Cost budget per account and cost center
budget_costs = unpivot_months(pd.read_excel(xlsx, sheet_name="Cost_Budget", dtype={"Account": str}),
                              ["Year", "Account", "Cost_Center"], "amount_pln")
budget_costs = budget_costs.rename(columns={"Year": "year", "Account": "account", "Cost_Center": "cost_center"})

assumptions = pd.read_excel(xlsx, sheet_name="Assumptions").rename(columns={
    "Year": "year", "EUR_PLN": "budget_eur_pln", "Gas_PLN_MWh": "budget_gas_pln_mwh",
    "Electricity_PLN_MWh": "budget_electricity_pln_mwh", "CO2_PLN_t": "budget_co2_pln_t"})

save_table(spark.createDataFrame(budget_sales)
           .withColumn("quantity_t", F.col("quantity_t").cast("decimal(18,3)"))
           .withColumn("price_per_t", F.col("price_per_t").cast("decimal(18,4)")),
           "silver_budget_sales")
save_table(spark.createDataFrame(budget_costs)
           .withColumn("amount_pln", F.col("amount_pln").cast("decimal(18,2)")),
           "silver_budget_costs")
save_table(spark.createDataFrame(assumptions), "silver_budget_assumptions")

## 4. Master data

In [ ]:
def read_master(file_name: str):
    df = spark.read.option("header", True).option("encoding", "UTF-8").csv(f"{SPARK_FILES}/master_data/{file_name}")
    return df.select([F.trim(F.col(c)).alias(c) for c in df.columns])


save_table(read_master("products.csv"), "silver_products")
save_table(read_master("customers.csv").withColumn("customer_id", F.upper("customer_id")), "silver_customers")
save_table(read_master("cost_centers.csv"), "silver_cost_centers")
save_table(read_master("accounts.csv").withColumn("pl_line_order", F.col("pl_line_order").cast("int")),
           "silver_accounts")

## 5. Daily FX rates with the "previous business day" rule

For VAT/CIT purposes Polish companies convert foreign-currency invoices at the NBP average rate
from the **last business day before the invoice date**. We build a calendar with one row per day:

* `nbp_mid_rate` – rate published on that day (null on weekends/holidays)
* `rate_for_invoice_date` – rate to use for an invoice dated that day

In [ ]:
fx = spark.read.table("bronze_nbp_fx_rates")

bounds = fx.agg(F.min("effective_date").alias("d0"), F.max("effective_date").alias("d1")).first()
calendar = spark.createDataFrame([(bounds.d0, bounds.d1)], ["d0", "d1"]).select(
    F.explode(F.sequence(F.col("d0"), F.date_add(F.col("d1"), 1))).alias("date"))

w_currency = Window.partitionBy("currency").orderBy("date")
w_fill = w_currency.rowsBetween(Window.unboundedPreceding, 0)

fx_daily = (calendar
    .crossJoin(fx.select("currency").distinct())
    .join(fx.withColumnRenamed("effective_date", "date"), ["currency", "date"], "left")
    .withColumn("last_mid_rate", F.last("mid_rate", ignorenulls=True).over(w_fill))
    .withColumn("last_table_no", F.last("table_no", ignorenulls=True).over(w_fill))
    # rate valid for an invoice dated D = last published rate up to D-1
    .withColumn("rate_for_invoice_date", F.lag("last_mid_rate", 1).over(w_currency))
    .withColumn("nbp_table_for_invoice", F.lag("last_table_no", 1).over(w_currency))
    .select("currency", "date", F.col("mid_rate").alias("nbp_mid_rate"), "table_no",
            "rate_for_invoice_date", "nbp_table_for_invoice")
    .filter(F.col("rate_for_invoice_date").isNotNull()))

save_table(fx_daily, "silver_fx_rates_daily")
display(fx_daily.orderBy("date").limit(14))